In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
df_train=pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')

In [ ]:
df_test=pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

In [ ]:
neighborhood_medians = (df_train.groupby("Neighborhood")["LotFrontage"].median())
df_train["LotFrontage"] = df_train["LotFrontage"].fillna(df_train["Neighborhood"].map(neighborhood_medians))
df_test["LotFrontage"] = df_test["LotFrontage"].fillna(df_test["Neighborhood"].map(neighborhood_medians))

In [ ]:
df_train["years_old"] =(df_train["YrSold"] - df_train["YearBuilt"]).clip(lower=0)
df_test["years_old"] = (df_test["YrSold"] - df_test["YearBuilt"]).clip(lower=0)
df_train['LotArea']=np.log1p(df_train['LotArea'])
df_test['LotArea']=np.log1p(df_test['LotArea'])
df_train["GarageYrBlt"] = df_train["GarageYrBlt"].fillna(df_train["YearBuilt"])
df_test["GarageYrBlt"] = df_test["GarageYrBlt"].fillna(df_test["YearBuilt"])
df_train["log_garage_age"] = np.log1p((df_train["YrSold"] - df_train["GarageYrBlt"]).clip(lower=0))
df_test["log_garage_age"] = np.log1p((df_test["YrSold"] - df_test["GarageYrBlt"]).clip(lower=0))
df_train.drop(columns=['GarageYrBlt','YrSold','YearBuilt'],inplace=True)
df_test.drop(columns=['GarageYrBlt','YrSold','YearBuilt'],inplace=True)
df_train['TotalBath'] = df_train['FullBath'].fillna(0) + (0.5 * df_train['HalfBath'].fillna(0)) + df_train['BsmtFullBath'].fillna(0) + (0.5 * df_train['BsmtHalfBath'].fillna(0))
df_train=df_train[(df_train['TotalBath']<5) & (df_train['SalePrice']>30000)]
df_train.drop(columns=['FullBath','HalfBath','BsmtFullBath','BsmtHalfBath'],inplace=True)
df_test['TotalBath'] = df_test['FullBath'].fillna(0) + (0.5 * df_test['HalfBath'].fillna(0)) + df_test['BsmtFullBath'].fillna(0) + (0.5 * df_test['BsmtHalfBath'].fillna(0))
df_test.drop(columns=['FullBath','HalfBath','BsmtFullBath','BsmtHalfBath'],inplace=True)

In [ ]:
X=df_train.drop(columns=['SalePrice'])
y=df_train['SalePrice']
df_train.drop(columns=['SalePrice'],inplace=True)

In [ ]:
df_train['MSSubClass']=df_train['MSSubClass'].astype(object)
df_test['MSSubClass']=df_test['MSSubClass'].astype(object)

In [ ]:
num_cols=df_train.select_dtypes(include=["int64", "float64"]).columns[df_train.select_dtypes(include=["int64", "float64"]).isnull().any()].tolist()
obj_cols=df_train.select_dtypes(include=["object"]).columns[df_train.select_dtypes(include=["object"]).isnull().any()].tolist()

In [ ]:
fill_rules={}
for col in num_cols:
    fill_rules[col]=df_train[col].median()
for col in obj_cols:
    fill_rules[col]="Not"

In [ ]:
df_train.fillna(value=fill_rules,inplace=True)

In [ ]:
num_cols_sub=df_test.select_dtypes(include=["int64", "float64"]).columns[df_test.select_dtypes(include=["int64", "float64"]).isnull().any()].tolist()
obj_cols_sub=df_test.select_dtypes(include=["object"]).columns[df_test.select_dtypes(include=["object"]).isnull().any()].tolist()

In [ ]:
fill_rules_sub={}
for col in num_cols_sub:
    fill_rules_sub[col]=df_train[col].median()
for col in obj_cols_sub:
    fill_rules_sub[col]="Not"

In [ ]:
df_test.fillna(value=fill_rules_sub,inplace=True)

In [ ]:
not_num_cols=['OverallQual','OverallCond','Id']
numerical_cols = df_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
numerical_cols_test=df_test.select_dtypes(include=["int64", "float64"]).columns.tolist()
numerical_cols=[col for col in numerical_cols if col not in not_num_cols]

In [ ]:
with open("/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt", "r", encoding="utf-8") as file:
    content = file.read()
    print(content)

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,StandardScaler
from sklearn.pipeline import make_pipeline

Categorical Hierarchial Columns:
* Utilities:[ELO,NoSeWa,NoSewr,AllPub]
* ExterQual: [Po,Fa,TA,Gd,Ex]
* ExterCond: [Po,Fa,TA,Gd,Ex]
* BsmtQual:[NaN,Po,Fa,TA,Gd,Ex]
* BsmtCond:[NaN,Po,Fa,TA,Gd,Ex]
* BsmtExposure: [NaN,No,Mn,Av,Gd]
* BsmtFinType1:[NaN,Unf,LwQ,Rec,BLQ,ALQ,GLQ]
* BsmtFinType2:[NaN,Unf,LwQ,Rec,BLQ,ALQ,GLQ]
* HeatingQC:[Po,Fa,TA,Gd,Ex]
* KitchenQual:[Po,Fa,TA,Gd,Ex]
* FireplaceQu:[NaN,Po,Fa,TA,Gd,Ex]
* GarageQual:[NaN,Po,Fa,TA,Gd,Ex]
* GarageCond:[NaN,Po,Fa,TA,Gd,Ex]
* PoolQC:[NaN,Fa,TA,Gd,Ex]
* Fence:[NaN,MnWw,GdWo,MnPrv,GdPrv]
       
       
       
  

In [ ]:
df_test.head()

In [ ]:
trf1=make_column_transformer((OrdinalEncoder(categories=[['Not','ELO','NoSeWa','NoSewr','AllPub']],handle_unknown="use_encoded_value",unknown_value=-1),['Utilities']),remainder='passthrough',verbose_feature_names_out=False)
trf2=make_column_transformer((OrdinalEncoder(categories=[['Not','Po','Fa','TA','Gd','Ex']]*4),['ExterQual','ExterCond','HeatingQC','KitchenQual']),remainder='passthrough',verbose_feature_names_out=False)
trf3=make_column_transformer((OrdinalEncoder(categories=[['Not','Po','Fa','TA','Gd','Ex']]*5,handle_unknown="use_encoded_value",unknown_value=-1),['BsmtQual','BsmtCond','FireplaceQu','GarageQual','GarageCond']),remainder='passthrough',verbose_feature_names_out=False)
trf4=make_column_transformer((OrdinalEncoder(categories=[['Not','Unf','LwQ','Rec','BLQ','ALQ','GLQ']]*2,handle_unknown="use_encoded_value",unknown_value=-1),['BsmtFinType1','BsmtFinType2']),remainder='passthrough',verbose_feature_names_out=False)
trf5=make_column_transformer((OrdinalEncoder(categories=[['Not','No','Mn','Av','Gd']],handle_unknown="use_encoded_value",unknown_value=-1),['BsmtExposure']),remainder='passthrough',verbose_feature_names_out=False)
trf6=make_column_transformer((OrdinalEncoder(categories=[['Not','Fa','TA','Gd','Ex']],handle_unknown="use_encoded_value",unknown_value=-1),['PoolQC']),remainder='passthrough',verbose_feature_names_out=False)
trf7=make_column_transformer((OrdinalEncoder(categories=[['Not','MnWw','GdWo','MnPrv','GdPrv']],handle_unknown="use_encoded_value",unknown_value=-1),['Fence']),remainder='passthrough',verbose_feature_names_out=False)
trf8=make_column_transformer((StandardScaler(),numerical_cols),remainder='passthrough',verbose_feature_names_out=False)

In [ ]:
pipe=make_pipeline(trf1,trf2,trf3,trf4,trf5,trf6,trf7,trf8)

In [ ]:
import sklearn
sklearn.set_config(transform_output="pandas")
df_train_processed=pipe.fit_transform(df_train)
df_test_processed=pipe.transform(df_test)

In [ ]:
df_train_processed['MSSubClass']=df_train_processed['MSSubClass'].astype(object)
df_test_processed['MSSubClass']=df_test_processed['MSSubClass'].astype(object)

In [ ]:
for cols in numerical_cols:
    if cols!='MSSubClass':
        df_train_processed[cols]=df_train_processed[cols].astype(float)
        df_test_processed[cols]=df_test_processed[cols].astype(float)

In [ ]:
obj_cols= df_train_processed.select_dtypes(include=["object"]).columns.tolist()

In [ ]:
obj_cols

In [ ]:
combined_df = pd.concat([df_train,df_test], ignore_index=True)
combined_df.drop(columns=['Id'],inplace=True)
combined_df['MSSubClass']=combined_df['MSSubClass'].astype(object)

In [ ]:
comb_df_trf=make_column_transformer((OneHotEncoder(sparse_output=False),obj_cols))
temp=comb_df_trf.fit_transform(combined_df)
comb_df_new=pd.DataFrame(temp,columns=comb_df_trf.get_feature_names_out(), index=combined_df.index)

In [ ]:
ohe_trf=make_column_transformer((OneHotEncoder(categories=comb_df_trf.named_transformers_["onehotencoder"].categories_,sparse_output=False),obj_cols),remainder='passthrough',verbose_feature_names_out=False)

In [ ]:
sklearn.set_config(transform_output="pandas")
pipe2=make_pipeline(ohe_trf)
df_train_final=pipe2.fit_transform(df_train_processed)
df_test_final=pipe2.transform(df_test_processed)

**Outlier Check:
* LotArea>60000
* LotFrontage>150
* BsmtFinSF1>2000
* BsmtFinSF2>500
* OpenPorchSF>350
* EnclosedPorch>200
* 3SsnPorch>200
* ScreenPorch>300


In [ ]:
from sklearn.tree import DecisionTreeRegressor

In [ ]:
from sklearn.model_selection import cross_val_score,train_test_split,GridSearchCV

In [ ]:
from sklearn.metrics import r2_score

In [ ]:
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor,AdaBoostRegressor

In [ ]:
y_log=np.log1p(y)

In [ ]:
params_rf={'n_estimators':[100],'max_features':[0.25,0.5,0.75,1.0],'max_samples':[0.25,0.5,1.0]}
rf=RandomForestRegressor()
cv_rf=GridSearchCV(rf,param_grid=params_rf,cv=5,verbose=3,scoring='r2')
cv_rf.fit(df_train_final,y_log)

In [ ]:
cv_rf.best_score_

In [ ]:
gb=GradientBoostingRegressor()
params_gb = {"n_estimators": [500],"max_depth": [3],"learning_rate":[0.001,0.01,0.1]}
cv_gb=GridSearchCV(gb,param_grid=params_gb,cv=5,verbose=3,scoring='neg_root_mean_squared_error')
cv_gb.fit(df_train_final,y_log)

In [ ]:
cv_gb.best_score_

In [ ]:
import xgboost as xgb
model=xgb.XGBRegressor(n_estimators=500,max_leaves=8)
cross_val_score(model,df_train_final,y_log,cv=5,scoring="neg_root_mean_squared_error").mean()

In [ ]:
gb=GradientBoostingRegressor(max_depth=3,n_estimators=500,learning_rate=0.1)
gb.fit(df_train_final,y_log)
df_submission=pd.DataFrame({'Id':df_test_final['Id'],'SalePrice':np.expm1(gb.predict(df_test_final))})

In [ ]:
gb=GradientBoostingRegressor()
param_grid={'max_depth':[3],'n_estimators':[500],'learning_rate':[0.1]}
gb.fit(df_train_final,y_log)
cv_gb=GridSearchCV(gb,cv=5,param_grid=param_grid,verbose=3,scoring='r2')
cv_gb.fit(df_train_final,y_log)
cv_gb.best_score_

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import GridSearchCV
lgbm = LGBMRegressor(random_state=42, verbosity=-1)
param_grid = {
    "n_estimators":[50,100,500],
    "learning_rate": [0.01, 0.1,1.0],
    "max_depth":[3,8,None],
}
grid_search_lgb = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",  # Optimizes for Log-RMSE
    n_jobs=-1,
    verbose=1,
)
grid_search_lgb.fit(df_train_final,y_log)

In [ ]:
grid_search_lgb.best_score_

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LassoCV, RidgeCV
from xgboost import XGBRegressor
estimators = [
    ("lgbm", LGBMRegressor(max_depth=3, n_estimators=500,verbosity=-1)),
    ("xgb", XGBRegressor(n_estimators=500,max_leaves=8)),
    ("gb", GradientBoostingRegressor(max_depth=3,n_estimators=500,learning_rate=0.1)),
]

stacking_regressor = StackingRegressor(
    estimators=estimators, final_estimator=RidgeCV(),n_jobs=-1
)
cross_val_score(stacking_regressor,df_train_final.values,y_log,cv=5,scoring="neg_root_mean_squared_error").mean()

In [ ]:
estimators = [
    ("lgbm", LGBMRegressor(max_depth=3, n_estimators=500,verbosity=-1)),
    ("xgb", XGBRegressor(n_estimators=500,max_leaves=8)),
    ("gb", GradientBoostingRegressor(max_depth=3,n_estimators=500,learning_rate=0.1)),  # Your current gradient boosting setup
]

stacking_regressor = StackingRegressor(
    estimators=estimators, final_estimator=RidgeCV(),n_jobs=-1
)
stacking_regressor.fit(df_train_final.values,y_log)
df_submission=pd.DataFrame({'Id':df_test_final['Id'],'SalePrice':np.expm1(stacking_regressor.predict(df_test_final.values))})

In [ ]:
df_submission['Id']=df_submission['Id'].astype(int)
import os
# Specify the name or path of the file you want to delete
file_to_remove = "submission.csv"
# Check if the file exists before attempting to delete
if os.path.exists(file_to_remove):
    os.remove(file_to_remove)
df_submission.to_csv("submission.csv", index=False)

In [ ]:
df_submission